# 2B.2 Recursieve backtracking — lees, voorspel, verbeter

Deze code is (met opzet) niet altijd correct. Lees elk fragment, **voorspel** wat het doet vooraleer je uitvoert, en zoek de fout als er een is.

Elke oefening gebruikt hetzelfde soort doolhof: een raster van `'.'` (vrij) en `'X'` (muur). We zoeken telkens een pad van linksboven `(0, 0)` naar rechtsonder `(3, 3)`, met stappen naar onder, boven, rechts en links.

## Fout 1: vergeten backtracken

Dit moet een pad tonen van `(0, 0)` naar `(3, 3)`, zonder dat er vakjes in het uiteindelijke pad staan die eigenlijk tot een doodlopend straatje behoorden.

**Antwoord:** Als een stap niet tot een oplossing leidt (`vind_uitweg(...)` geeft `False` terug), wordt die stap nooit uit `ontsnapping` verwijderd. Omdat `check_plaats` gebruikt om te bepalen of een vakje al bezocht is (`(x, y) not in ontsnapping`), blijven doodlopende vakjes voor altijd als 'bezocht' meetellen. Het algoritme vindt uiteindelijk wel het einde, maar `ontsnapping` bevat dan nog steeds de vakjes van elke mislukte zijsprong (bv. `(1, 1)`, `(0, 1)`, `(0, 2)`). Die horen niet in het finale pad thuis. Voeg een `ontsnapping.pop()` toe wanneer de recursieve aanroep `False` teruggeeft, zodat je backtrackt naar de vorige stap.

In [2]:
doolhof = [
    ['.', '.', '.', 'X'],
    ['.', '.', 'X', 'X'],
    ['.', '.', '.', '.'],
    ['.', '.', '.', '.'],
]

def check_plaats(x, y, ontsnapping):
    return (0 <= x < len(doolhof) and 0 <= y < len(doolhof[0])
            and doolhof[x][y] == '.' and (x, y) not in ontsnapping)

def vind_uitweg(x, y, ontsnapping):
    ontsnapping.append((x, y))
    if (x, y) == (3, 3):
        return True
    for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
        nx, ny = x + dx, y + dy
        if check_plaats(nx, ny, ontsnapping):
            if vind_uitweg(nx, ny, ontsnapping):
                return True
            else:
                ontsnapping.pop()
    return False

ontsnapping = []
gevonden = vind_uitweg(0, 0, ontsnapping)
print('Pad gevonden:', gevonden)
print('Pad:', ontsnapping)

Pad gevonden: True
Pad: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (2, 1), (2, 2), (3, 2), (3, 3)]

## Fout 2: return-waarde van de recursieve aanroep niet gebruikt

Dit moet `True` teruggeven, want er bestaat een pad van `(0, 0)` naar `(3, 3)` in dit doolhof.

**Antwoord:** In de lus wordt `vind_uitweg(nx, ny, ontsnapping)` wel opgeroepen, maar het resultaat ervan (`True`/`False`) wordt nergens gebruikt. Zelfs als een diepere aanroep het eindpunt bereikt en `True` teruggeeft, gaat de huidige functie gewoon door met de rest van de lus (en probeert dus nog andere, nutteloze richtingen) om uiteindelijk toch altijd zelf `False` terug te geven. Het succes 'verdwijnt' dus ergens onderweg naar boven toe. Je moet controleren of de recursieve aanroep `True` teruggaf en dat dan zelf ook meteen doorgeven met `return True`.

In [4]:
doolhof = [
    ['.', '.', '.', 'X'],
    ['.', '.', 'X', 'X'],
    ['.', '.', '.', '.'],
    ['.', '.', '.', '.'],
]

def check_plaats(x, y, ontsnapping):
    return (0 <= x < len(doolhof) and 0 <= y < len(doolhof[0])
            and doolhof[x][y] == '.' and (x, y) not in ontsnapping)

def vind_uitweg(x, y, ontsnapping):
    ontsnapping.append((x, y))
    if (x, y) == (3, 3):
        return True
    for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
        nx, ny = x + dx, y + dy
        if check_plaats(nx, ny, ontsnapping):
            if vind_uitweg(nx, ny, ontsnapping):
                return True
    ontsnapping.pop()
    return False

ontsnapping = []
gevonden = vind_uitweg(0, 0, ontsnapping)
print('Pad gevonden:', gevonden)
print('Pad:', ontsnapping)

Pad gevonden: True
Pad: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (2, 1), (2, 2), (3, 2), (3, 3)]

## Fout 3: foute grenscontrole

Dit moet, net zoals hierboven, controleren of een stap binnen het doolhof blijft.

**Antwoord:** De vergelijkingen gebruiken `<=` in plaats van `<`: `0 <= x <= len(doolhof)`. Geldige indices lopen van `0` tot en met `len(doolhof) - 1`, dus `x == len(doolhof)` (of `y == len(doolhof[0])`) wordt hier onterecht als geldig beschouwd. Zodra zo'n stap uitgeprobeerd wordt wil de code `doolhof[x][y]` opvragen op een index die niet bestaat, wat een `IndexError` geeft. De grenscontrole moet een strikte `<` gebruiken.

In [6]:
doolhof = [
    ['.', '.', '.', 'X'],
    ['.', '.', 'X', 'X'],
    ['.', '.', '.', '.'],
    ['.', '.', '.', '.'],
]

def check_plaats(x, y, ontsnapping):
    return (0 <= x < len(doolhof) and 0 <= y < len(doolhof[0])
            and doolhof[x][y] == '.' and (x, y) not in ontsnapping)

def vind_uitweg(x, y, ontsnapping):
    ontsnapping.append((x, y))
    if (x, y) == (3, 3):
        return True
    for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
        nx, ny = x + dx, y + dy
        if check_plaats(nx, ny, ontsnapping):
            if vind_uitweg(nx, ny, ontsnapping):
                return True
            else:
                ontsnapping.pop()
    return False

ontsnapping = []
print(vind_uitweg(0, 0, ontsnapping))

True

## Fout 4: vergeten controleren of een vakje al bezocht is

Dit moet, net zoals hierboven, controleren of een vakje geldig is om naartoe te stappen.

**Antwoord:** `check_plaats` controleert niet meer of `(x, y)` al in `ontsnapping` zit. Daardoor kan de recursie eindeloos heen en weer springen tussen twee aangrenzende vrije vakjes (bv. steeds terug naar het vakje waar je net vandaan kwam), want elk van beide vakjes blijft altijd 'geldig' om naartoe te stappen. Dit geeft nooit een basisoplossing en leidt tot een `RecursionError`. De voorwaarde `(x, y) not in ontsnapping` is net zo essentieel als de grenscontrole.

In [8]:
doolhof = [
    ['.', '.', '.', 'X'],
    ['.', '.', 'X', 'X'],
    ['.', '.', '.', '.'],
    ['.', '.', '.', '.'],
]

def check_plaats(x, y, ontsnapping):
    return (0 <= x < len(doolhof) and 0 <= y < len(doolhof[0])
            and doolhof[x][y] == '.' and (x, y) not in ontsnapping)

def vind_uitweg(x, y, ontsnapping):
    ontsnapping.append((x, y))
    if (x, y) == (3, 3):
        return True
    for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
        nx, ny = x + dx, y + dy
        if check_plaats(nx, ny, ontsnapping):
            if vind_uitweg(nx, ny, ontsnapping):
                return True
            else:
                ontsnapping.pop()
    return False

ontsnapping = []
print(vind_uitweg(0, 0, ontsnapping))

True